# Proyecto IA: Brecha Digital y Rendimiento Escolar en el Perú (2025)

**FASE 1 — Limpieza e integración ENAHO + ESCALE (Jupyter Notebook)**

Este notebook contiene el procedimiento reproducible para pasar de seis archivos oficiales a **un único dataset maestro**. Los resultados se producen en el momento de ejecutar las celdas; no hace falta ejecutar modelos de IA todavía.

**Antes de empezar:**
1. Abre una terminal en la carpeta del proyecto y ejecuta `pip install -r requirements.txt` (si todavía no tienes estas bibliotecas).
2. Inicia Jupyter con `jupyter notebook` y abre `notebooks/01_limpieza_enaho_escale_2025.ipynb`.
3. Verifica que la carpeta `data/raw` contiene los cuatro CSV y los dos DBF incluidos en el ZIP.
4. En Jupyter selecciona **Kernel → Restart & Run All**, o ejecuta cada celda en orden con **Shift+Enter**.

**Control metodológico:** claves textuales, sin filas duplicadas, decimales corregidos, cruces relacionales validados. No hacemos imputación, OneHotEncoding o escalamiento global para evitar fuga de información.


## Paso 1. Preparar el entorno

Comprueba que los seis archivos estén en `data/raw`. Las llaves se cargarán como texto para conservar ceros a la izquierda.


In [8]:
from pathlib import Path
import json
import struct
import pandas as pd
import numpy as np

# Detectar la raíz del proyecto
ROOT = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path.cwd().resolve()

# Si Jupyter se ejecutó dentro de la carpeta 'notebooks', sube un nivel a la raíz
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# Modificado: Ahora RAW apunta directamente a la raíz donde están tus archivos sueltos
RAW = ROOT 
PROCESSED = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reportes'

# Crear carpetas de salida si no existen
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# Diccionario de archivos tal como están en tu directorio
ARCHIVOS = {
    'enaho100': 'Enaho01-2025-100.csv',
    'enaho200': 'Enaho01-2025-200.csv',
    'enaho300': 'Enaho01A-2025-300.csv',
    'sumaria': 'Sumaria-2025.csv',
    'padron': 'Padron.dbf',
    'padlocal': 'Padlocal.dbf'
}

# Verificación y lectura de archivos
for etiqueta, nombre in ARCHIVOS.items():
    ruta = RAW / nombre
    if not ruta.exists():
        raise FileNotFoundError(f'No se encontró el archivo {nombre} en la ruta: {RAW}')
    print(f'OK {etiqueta}: {nombre} ({ruta.stat().st_size / (1024**2):.1f} MiB)')



OK enaho100: Enaho01-2025-100.csv (34.1 MiB)
OK enaho200: Enaho01-2025-200.csv (15.0 MiB)
OK enaho300: Enaho01A-2025-300.csv (105.7 MiB)
OK sumaria: Sumaria-2025.csv (22.3 MiB)
OK padron: Padron.dbf (137.2 MiB)
OK padlocal: Padlocal.dbf (45.2 MiB)


## Paso 2. Crear funciones de lectura segura

Los CSV de ENAHO usan `sep=";"`, `encoding="latin1"` y comas decimales. Los archivos DBF se leen sin instalar bibliotecas adicionales.


In [9]:
LLAVES_PERSONA = ['AÑO', 'CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO', 'UBIGEO']
LLAVES_HOGAR = ['AÑO', 'CONGLOME', 'VIVIENDA', 'HOGAR', 'UBIGEO']

def csv_inei(archivo, columnas):
    archivo = RAW / archivo
    columnas = list(dict.fromkeys(columnas))
    encabezado = pd.read_csv(archivo, sep=';', encoding='latin1', nrows=0).columns
    encontrados = set(x.upper().strip() for x in encabezado)
    faltan = set(columnas) - encontrados
    if faltan:
        raise ValueError(f'{archivo.name}: columnas no encontradas: {sorted(faltan)}')
    tabla = pd.read_csv(
        archivo, sep=';', encoding='latin1',
        usecols=lambda x: x.upper().strip() in columnas,
        dtype=str, keep_default_na=False, low_memory=False
    )
    tabla.columns = tabla.columns.str.upper().str.strip()
    for columna in tabla.columns:
        tabla[columna] = tabla[columna].astype('string').str.strip().replace('', pd.NA)
    return tabla

def dbf_columnas(archivo, columnas):
    """Lector mínimo DBF, sin paquetes externos; conserva CODGEO y CODLOCAL como texto."""
    archivo = RAW / archivo
    columnas = list(dict.fromkeys(columnas))
    with archivo.open('rb') as f:
        cabecera = f.read(32)
        n_registros = struct.unpack('<I', cabecera[4:8])[0]
        largo_cabecera = struct.unpack('<H', cabecera[8:10])[0]
        largo_registro = struct.unpack('<H', cabecera[10:12])[0]
        campos = {}
        offset = 1  # primer byte = marca de registro eliminado
        while f.tell() < largo_cabecera:
            info = f.read(32)
            if not info or info[0] == 13:
                break
            nombre = info[:11].split(b'\0', 1)[0].decode('latin1').upper().strip()
            nbytes = info[16]
            if nombre in columnas:
                campos[nombre] = (offset, offset + nbytes)
            offset += nbytes
        faltantes = set(columnas) - set(campos)
        if faltantes:
            raise ValueError(f'{archivo.name}: faltan {sorted(faltantes)}')
        f.seek(largo_cabecera)
        registros = []
        for _ in range(n_registros):
            linea = f.read(largo_registro)
            if len(linea) != largo_registro:
                raise IOError(f'{archivo.name}: registro truncado')
            if linea[:1] == b'*':
                continue
            registros.append({nombre: linea[a:b].decode('latin1', errors='replace').strip() or None
                              for nombre, (a, b) in campos.items()})
    return pd.DataFrame.from_records(registros, columns=columnas).astype('string')

def numero(columna):
    return pd.to_numeric(columna.astype('string').str.replace(',', '.', regex=False), errors='coerce')

def validar_clave(tabla, columnas, nombre):
    faltan = int(tabla[columnas].isna().any(axis=1).sum())
    duplicados = int(tabla.duplicated(columnas).sum())
    if faltan or duplicados:
        raise ValueError(f'{nombre}: claves incompletas={faltan}, duplicadas={duplicados}')
    print(f'{nombre}: claves comprobadas; {len(tabla):,} filas, 0 duplicados')


## Paso 3. Leer ENAHO y filtrar edades

Importamos los módulos 100, 200, 300 y Sumaria; comprobamos claves duplicadas y seleccionamos personas de 6 a 17 años.


In [10]:
p200 = csv_inei(ARCHIVOS['enaho200'], LLAVES_PERSONA + ['MES','P203','P207','P208A','NCONGLOME'])
p300_cols = ['P300A','P306','P307','P308A','P308B','P314A'] + [f'P314B${i}' for i in range(1,8)] + [f'P316${i}' for i in range(1,13)]
p300 = csv_inei(ARCHIVOS['enaho300'], LLAVES_PERSONA + p300_cols)
p100 = csv_inei(ARCHIVOS['enaho100'], LLAVES_HOGAR + ['ESTRATO','DOMINIO','P1121','P1144'])
sumaria = csv_inei(ARCHIVOS['sumaria'], LLAVES_HOGAR + ['POBREZA','INGHOG2D','GASHOG2D','FACTOR07','MIEPERHO'])

for nombre, tabla, pk in [
    ('ENAHO 200',p200,LLAVES_PERSONA), ('ENAHO 300',p300,LLAVES_PERSONA),
    ('ENAHO 100',p100,LLAVES_HOGAR), ('Sumaria',sumaria,LLAVES_HOGAR)]:
    validar_clave(tabla,pk,nombre)

p200['EDAD'] = numero(p200['P208A'])
base = p200.loc[p200['EDAD'].between(6,17)].copy()
print('Población de 6 a 17 años:', len(base))


ENAHO 200: claves comprobadas; 115,145 filas, 0 duplicados
ENAHO 300: claves comprobadas; 104,446 filas, 0 duplicados
ENAHO 100: claves comprobadas; 44,599 filas, 0 duplicados
Sumaria: claves comprobadas; 33,702 filas, 0 duplicados
Población de 6 a 17 años: 24195


## Paso 4. Unir los módulos de ENAHO

Un registro por persona. Se excluyen las observaciones sin módulo 300, y se controla la correspondencia de hogares.


In [11]:
# Intersección con 300: excluir SOLO los que no tienen registro educativo;
# nunca inventar contenido faltante ni repetir hogares/personas.
registros_200 = len(base)
base = base.merge(p300, on=LLAVES_PERSONA, how='left', validate='one_to_one', indicator='_cruce_300')
no_educacion = int((base['_cruce_300'] != 'both').sum())
base = base.loc[base['_cruce_300'] == 'both'].drop(columns=['_cruce_300'])
base = base.merge(p100, on=LLAVES_HOGAR, how='left', validate='many_to_one', indicator='_cruce_100')
no_vivienda = int((base['_cruce_100'] != 'both').sum())
base = base.drop(columns=['_cruce_100'])
base = base.merge(sumaria, on=LLAVES_HOGAR, how='left', validate='many_to_one', indicator='_cruce_sumaria')
no_sumaria = int((base['_cruce_sumaria'] != 'both').sum())
base = base.drop(columns=['_cruce_sumaria'])
if no_vivienda or no_sumaria:
    raise AssertionError('La fusión no cubre todos los hogares; revisar datos antes de continuar.')
assert len(base) == registros_200 - no_educacion
validar_clave(base, LLAVES_PERSONA, 'Base ENAHO integrada')
print('Sin módulo 300 (excluidos):',no_educacion)
print('Personas ENAHO integradas:',len(base))


Base ENAHO integrada: claves comprobadas; 23,645 filas, 0 duplicados
Sin módulo 300 (excluidos): 550
Personas ENAHO integradas: 23645


## Paso 5. Leer los dos DBF de ESCALE

Usamos `TALUM`, `TDOC`, `TSEC` (nombres reales del DBF); trabajamos con primaria y secundaria para la población objetivo.


In [12]:
campos_padron = ['COD_MOD','CODLOCAL','CODGEO','NIV_MOD','D_NIV_MOD','GESTION','AREA_CENSO','DAREACENSO','NLAT_IE','NLONG_IE','TALUM','TDOC','TSEC']
campos_local = ['CODLOCAL','CODGEO','AREA_LOC']
padron = dbf_columnas(ARCHIVOS['padron'],campos_padron)
locales = dbf_columnas(ARCHIVOS['padlocal'],campos_local)
validar_clave(locales,['CODLOCAL'],'Padlocal')
# La población objetivo es 6–17 años: únicamente EBR Primaria (B0) y Secundaria (F0).
padron_ebr = padron.loc[padron['NIV_MOD'].isin(['B0','F0'])].copy()
padron_ebr = padron_ebr.merge(
    locales.rename(columns={'CODGEO':'CODGEO_LOCAL'}),
    on='CODLOCAL', how='left', validate='many_to_one', indicator='_tiene_local'
)
geo_distinto = int(((padron_ebr['CODGEO'] != padron_ebr['CODGEO_LOCAL']) & padron_ebr['CODGEO_LOCAL'].notna()).sum())
padron_ebr = padron_ebr.drop(columns=['_tiene_local'])
for c in ['TALUM','TDOC','TSEC','NLAT_IE','NLONG_IE']:
    padron_ebr[c] = numero(padron_ebr[c])
# No inventar coordenadas cuando no existan o queden fuera de rangos de Perú.
padron_ebr.loc[~padron_ebr['NLAT_IE'].between(-19,0),'NLAT_IE'] = np.nan
padron_ebr.loc[~padron_ebr['NLONG_IE'].between(-82,-68),'NLONG_IE'] = np.nan
padron_ebr['PUBLICO'] = padron_ebr['GESTION'].isin(['1','2']).astype(int)
padron_ebr['PRIVADO'] = padron_ebr['GESTION'].eq('3').astype(int)
print('Locales escolares:',len(locales))
print('Instituciones primaria/secundaria:',len(padron_ebr))
print('Inconsistencias de UBIGEO entre padrón y local:',geo_distinto)


Padlocal: claves comprobadas; 69,642 filas, 0 duplicados
Locales escolares: 69642
Instituciones primaria/secundaria: 54038
Inconsistencias de UBIGEO entre padrón y local: 1


## Paso 6. Calcular indicadores escolares

Agrupamos instituciones a nivel de distrito y distrito/área. Nunca unimos escuelas individuales directamente a estudiantes.


In [13]:
def agregar_escale(tabla, columnas_grupo, prefijo):
    d = tabla.groupby(columnas_grupo,dropna=True).agg(
        N_COLEGIOS=('COD_MOD','size'),
        N_PUBLICOS=('PUBLICO','sum'),
        N_PRIVADOS=('PRIVADO','sum'),
        TOTAL_ALUMNOS=('TALUM','sum'),
        TOTAL_DOCENTES=('TDOC','sum'),
        TOTAL_SECCIONES=('TSEC','sum'),
        LAT_MEDIA=('NLAT_IE','mean'),
        LON_MEDIA=('NLONG_IE','mean')
    ).reset_index()
    d['RATIO_ALUMNOS_DOCENTE'] = d['TOTAL_ALUMNOS'].div(d['TOTAL_DOCENTES'].replace(0,np.nan))
    d['PROP_PUBLICOS'] = d['N_PUBLICOS'].div(d['N_COLEGIOS'])
    d['PROP_PRIVADOS'] = d['N_PRIVADOS'].div(d['N_COLEGIOS'])
    return d.rename(columns={c:prefijo+c for c in d.columns if c not in columnas_grupo})

distrito = agregar_escale(padron_ebr,['CODGEO'],'ESCALE_DIST_')
distrito_area = agregar_escale(padron_ebr,['CODGEO','AREA_CENSO'],'ESCALE_AREA_')
# ESTRATO 1–5: ámbito urbano; 6–8: ámbito rural (regla operativa; validar con diccionario INEI).
mapa_area = {**{str(i):'1' for i in range(1,6)}, **{str(i):'2' for i in range(6,9)}}
base['AREA_OPERATIVA_ENAHO'] = base['ESTRATO'].map(mapa_area)
base = base.merge(distrito, left_on='UBIGEO',right_on='CODGEO',how='left',validate='many_to_one')
base = base.drop(columns=['CODGEO'])
base = base.merge(distrito_area,left_on=['UBIGEO','AREA_OPERATIVA_ENAHO'],right_on=['CODGEO','AREA_CENSO'],how='left',validate='many_to_one')
base = base.drop(columns=['CODGEO','AREA_CENSO'])
base['TIENE_ESCALE_DISTRITO'] = base['ESCALE_DIST_N_COLEGIOS'].notna()
base['TIENE_ESCALE_AREA'] = base['ESCALE_AREA_N_COLEGIOS'].notna()
print('Distritos ESCALE EBR:',len(distrito))
print('Estudiantes con ESCALE distrito:',int(base['TIENE_ESCALE_DISTRITO'].sum()),'/',len(base))
print('Estudiantes con ESCALE área:',int(base['TIENE_ESCALE_AREA'].sum()),'/',len(base))


Distritos ESCALE EBR: 1891
Estudiantes con ESCALE distrito: 23645 / 23645
Estudiantes con ESCALE área: 23210 / 23645


## Paso 7. Convertir columnas numéricas y validar

Calculamos ingreso/gasto por miembro y comprobamos unicidad, año, edades, cobertura geográfica y tipos; no imputamos ni escalamos aún.


In [14]:
for c in ['INGHOG2D','GASHOG2D','FACTOR07','MIEPERHO']:
    base[c] = numero(base[c])
base['INGRESO_POR_MIEMBRO'] = base['INGHOG2D'].div(base['MIEPERHO'].replace(0,np.nan))
base['GASTO_POR_MIEMBRO'] = base['GASHOG2D'].div(base['MIEPERHO'].replace(0,np.nan))
# Nunca imputar, escalar ni codificar por toda la tabla: en ML se ajustará dentro de cada fold.
# P1144 y P314A se guardan como códigos originales; no convertir a booleanos sin diccionario.
base = base.sort_values(LLAVES_PERSONA).reset_index(drop=True)
validar_clave(base,LLAVES_PERSONA,'Dataset maestro final')
assert base['EDAD'].between(6,17).all()
assert base['AÑO'].eq('2025').all()
assert base['UBIGEO'].str.len().eq(6).all(), 'UBIGEO incompleto, revisar la carga.'
assert base['TIENE_ESCALE_DISTRITO'].mean() > 0.95, 'Cobertura ESCALE baja, revisar llave UBIGEO.'


Dataset maestro final: claves comprobadas; 23,645 filas, 0 duplicados


## Paso 8. Exportar el dataset maestro

Creamos `data/processed/dataset_enaho_escale_2025.csv`, diccionario de variables y reporte de auditoría; Parquet solo cuando esté disponible.


In [15]:
salida_csv = PROCESSED / 'dataset_enaho_escale_2025.csv'
base.to_csv(salida_csv,index=False,encoding='utf-8-sig')
# Si se dispone de pyarrow, exportar también Parquet (no es requisito para CSV).
try:
    base.to_parquet(PROCESSED/'dataset_enaho_escale_2025.parquet',index=False)
    print('Parquet generado.')
except (ImportError,ValueError,TypeError) as ex:
    print('Parquet opcional no generado; CSV completo y disponible:',type(ex).__name__)

descripciones = {
    'AÑO':'Año de ENAHO; llave textual','MES':'Mes de entrevista; llave textual',
    'CONGLOME':'Conglomerado INEI; grupo obligatorio para separar train/test',
    'VIVIENDA':'Identificador de vivienda, conservando ceros a la izquierda',
    'HOGAR':'Identificador de hogar, conservando ceros a la izquierda',
    'CODPERSO':'Identificador de persona, conservando ceros a la izquierda',
    'UBIGEO':'Código de distrito INEI de seis dígitos',
    'P208A':'Edad original del módulo ENAHO 200', 'EDAD':'Edad numérica filtrada de 6 a 17 años',
    'P207':'Sexo codificado; requiere diccionario ENAHO',
    'P203':'Parentesco en el hogar codificado; requiere diccionario',
    'P300A':'Lengua materna codificada; requiere diccionario',
    'P306':'Matrícula escolar codificada; requiere diccionario',
    'P307':'Asistencia escolar codificada; requiere diccionario',
    'P308A':'Nivel educativo codificado; requiere diccionario',
    'P308B':'Grado educativo codificado; requiere diccionario',
    'P314A':'Uso de internet codificado; requiere diccionario',
    'P1121':'Alumbrado/servicio del hogar según guía; verificar código',
    'P1144':'Internet del hogar según guía; verificar código y categoría',
    'ESTRATO':'Estrato de ENAHO', 'DOMINIO':'Dominio geográfico ENAHO',
    'POBREZA':'Situación de pobreza codificada; 1 extremo, 2 no extremo, 3 no pobre (guía)',
    'MIEPERHO':'Miembros por hogar en Sumaria',
    'INGHOG2D':'Ingreso del hogar de Sumaria, convertido de coma decimal',
    'GASHOG2D':'Gasto del hogar de Sumaria, convertido de coma decimal',
    'FACTOR07':'Factor de expansión de Sumaria',
    'INGRESO_POR_MIEMBRO':'INGHOG2D / MIEPERHO (sin imputación)',
    'GASTO_POR_MIEMBRO':'GASHOG2D / MIEPERHO (sin imputación)',
    'AREA_OPERATIVA_ENAHO':'1 urbano, 2 rural, aproximado desde ESTRATO (validar catálogo INEI)',
    'TIENE_ESCALE_DISTRITO':'Indica si el UBIGEO tiene instituciones EBR en ESCALE',
    'TIENE_ESCALE_AREA':'Indica si hubo coincidencia por UBIGEO y área operativa',
}
for campo in base.columns:
    if campo.startswith('ESCALE_DIST_'):
        descripciones[campo] = 'Indicador agregado de primaria/secundaria por distrito: '+campo.removeprefix('ESCALE_DIST_')
    elif campo.startswith('ESCALE_AREA_'):
        descripciones[campo] = 'Indicador agregado de primaria/secundaria por distrito y área: '+campo.removeprefix('ESCALE_AREA_')
    elif campo.startswith('P314B$'):
        descripciones[campo] = 'Código original del componente de uso de internet: '+campo
    elif campo.startswith('P316$'):
        descripciones[campo] = 'Código original del componente frecuencia/uso TIC: '+campo
    elif campo == 'NCONGLOME':
        descripciones[campo] = 'Identificador alternativo de conglomerado (ENAHO 200)'
diccionario = pd.DataFrame([
    {'variable':c,'tipo':str(base[c].dtype),'porcentaje_nulos':round(float(base[c].isna().mean()*100),2),
     'descripcion':descripciones.get(c,'Código original de ENAHO; revisar diccionario antes de usar')}
    for c in base.columns
])
diccionario.to_csv(REPORTS/'diccionario_variables.csv',index=False,encoding='utf-8-sig')

reporte = {
    'anio':'2025','fuentes':['ENAHO100','ENAHO200','ENAHO300','Sumaria','Padron','Padlocal'],
    'personas_6_a_17_modulo200':int(registros_200),
    'excluidos_sin_modulo300':int(no_educacion),
    'sin_match_modulo100':int(no_vivienda),
    'sin_match_sumaria':int(no_sumaria),
    'filas_dataset_maestro':int(len(base)),
    'columnas_dataset_maestro':int(len(base.columns)),
    'duplicados_clave_persona':int(base.duplicated(LLAVES_PERSONA).sum()),
    'distritos_enaho':int(base['UBIGEO'].nunique()),
    'conglomerados_enaho':int(base['CONGLOME'].nunique()),
    'distritos_escale_ebr':int(len(distrito)),
    'colegios_escale_ebr':int(len(padron_ebr)),
    'filas_con_escale_distrito':int(base['TIENE_ESCALE_DISTRITO'].sum()),
    'filas_con_escale_area':int(base['TIENE_ESCALE_AREA'].sum()),
    'desacuerdos_ubigeo_padron_padlocal':int(geo_distinto),
    'observaciones':[
        'Los CSV originales usan separador punto y coma y decimales con coma.',
        'Las llaves y UBIGEO se mantienen en formato string.',
        'Se conserva un registro por estudiante de 6 a 17 años con ENAHO 200 y 300.',
        'Padron.dbf tiene TALUM/TDOC/TSEC; no TALUMNO/TDOCENTE/TSECCION como la guía.',
        'No hay columna de altitud ni conectividad escolar en los dos DBF adjuntos.',
        'No se imputó ni escaló a nivel global; se hará dentro de cada fold en ML.',
        'P1144, P314A y los códigos escolares requieren contraste con el diccionario INEI.',
        'La codificación urbano-rural derivada de ESTRATO requiere verificación con el catálogo oficial.'
    ]
}
(REPORTS/'control_limpieza_2025.json').write_text(json.dumps(reporte,ensure_ascii=False,indent=2),encoding='utf-8')
print('\n================= RESULTADO =================')
for k,v in reporte.items():
    if k != 'observaciones': print(f'{k}: {v}')
print('CSV maestro:',salida_csv)
print('Diccionario:',REPORTS/'diccionario_variables.csv')
print('Reporte:',REPORTS/'control_limpieza_2025.json')
print('¡FIN DE LA FASE 1!')


Parquet generado.

================= RESULTADO =================
anio: 2025
fuentes: ['ENAHO100', 'ENAHO200', 'ENAHO300', 'Sumaria', 'Padron', 'Padlocal']
personas_6_a_17_modulo200: 24195
excluidos_sin_modulo300: 550
sin_match_modulo100: 0
sin_match_sumaria: 0
filas_dataset_maestro: 23645
columnas_dataset_maestro: 73
duplicados_clave_persona: 0
distritos_enaho: 1240
conglomerados_enaho: 5049
distritos_escale_ebr: 1891
colegios_escale_ebr: 54038
filas_con_escale_distrito: 23645
filas_con_escale_area: 23210
desacuerdos_ubigeo_padron_padlocal: 1
CSV maestro: C:\Users\Russbell\Downloads\inteligenciaartificial\datamodulos\data_limpia\data\processed\dataset_enaho_escale_2025.csv
Diccionario: C:\Users\Russbell\Downloads\inteligenciaartificial\datamodulos\data_limpia\reportes\diccionario_variables.csv
Reporte: C:\Users\Russbell\Downloads\inteligenciaartificial\datamodulos\data_limpia\reportes\control_limpieza_2025.json
¡FIN DE LA FASE 1!


## Comprobación visual final

La tabla debe mostrar **23.645 filas y 73 columnas** con el conjunto de archivos original. Si los resultados no coinciden, revisa los conteos impresos arriba. El CSV maestro está en `data/processed/`; usa siempre `dtype` textual cuando vuelvas a leer las llaves, porque Excel puede eliminar ceros.


In [16]:
print('Dimensiones:', base.shape)
print('Duplicados de estudiantes:',base.duplicated(LLAVES_PERSONA).sum())
print('Cobertura distrital ESCALE:',f"{base['TIENE_ESCALE_DISTRITO'].mean():.1%}")
display(base[['UBIGEO','CONGLOME','VIVIENDA','HOGAR','CODPERSO','EDAD','P308A','P308B','ESCALE_DIST_RATIO_ALUMNOS_DOCENTE']].head(10))


Dimensiones: (23645, 73)
Duplicados de estudiantes: 0
Cobertura distrital ESCALE: 100.0%


,UBIGEO,CONGLOME,VIVIENDA,HOGAR,CODPERSO,EDAD,P308A,P308B,ESCALE_DIST_RATIO_ALUMNOS_DOCENTE
0,010101,015001,113,11,03,16,3,5,17.057377
1,010101,015001,113,11,04,13,3,2,17.057377
2,010101,015001,130,11,03,9,2,0,17.057377
3,010101,015003,041,11,04,17,<NA>,<NA>,17.057377
4,010101,015003,116,11,04,9,2,0,17.057377
5,010101,015003,144,11,04,12,3,1,17.057377
6,010101,015004,003,11,03,11,2,0,17.057377
7,010101,015004,019,11,04,14,3,2,17.057377
8,010101,015004,034,11,04,6,2,0,17.057377
9,010101,015004,048,11,03,14,3,2,17.057377
